# E1 v3 Phase B - LR-L2 by feature-selection family

Run one Kaggle session for one `CLASSIFICATION_GROUP` across all five cohorts. This notebook evaluates fixed Phase-A panels with LR-L2 only; it never re-runs feature selection. Create three Kaggle versions in total: `filter`, `embedded`, and `wrappers`.

**Protocol override requested by the project owner:** wrapper panels are evaluated with 5 repeats x 5 folds on all five cohorts. This differs from the earlier 3x5/three-cohort wrapper plan and is recorded in `config.json`.

In [ ]:
from pathlib import Path
import json, os, shutil, subprocess, sys

INPUT_V3_DATASET = Path('/kaggle/input/experiment-data-v3')
PANEL_INPUT_ROOT = Path('/kaggle/input/e1-feature-selection-v3/E1_feature_selection_v3')
REPO_URL = 'https://github.com/datachain-uit/RNA-Seq-FS-Survey.git'
REPO_BRANCH = 'survey-v2-code'
WORK_ROOT = Path('/kaggle/working/kltn_data')

# One Kaggle version runs one FS group across every cohort.
# Create three versions total by changing only CLASSIFICATION_GROUP.
CLASSIFICATION_GROUP = 'filter'  # filter | embedded | wrappers
COHORTS = ['tcga_luad_lusc_primary_tumor', 'gse10245', 'gse30219', 'gse37745', 'gse42127']
REPEATS = [0, 1, 2, 3, 4]
K_GRID = [1, 2, 3, 4, 5, 10, 25, 50]

METHODS_BY_GROUP = {
    'filter': ['f_test', 'mutual_info', 'mrmr', 'relieff'],
    'embedded': ['l1_lr', 'elastic_net', 'rf_importance', 'svm_rfe'],
    'wrappers': ['ga', 'bpso'],
}
assert CLASSIFICATION_GROUP in METHODS_BY_GROUP
METHODS = METHODS_BY_GROUP[CLASSIFICATION_GROUP]
for cohort in COHORTS:
    assert PANEL_INPUT_ROOT.joinpath(CLASSIFICATION_GROUP, cohort, 'selected_genes.csv').is_file(), cohort
print({'family': CLASSIFICATION_GROUP, 'cohorts': COHORTS, 'methods': METHODS, 'repeats': REPEATS, 'k_grid': K_GRID})


In [ ]:
# Prepare an isolated, writable Kaggle work directory.
# Do not install requirements.txt in Kaggle: it pins the local-workstation stack
# and can replace Kaggle's matching NumPy/SciPy wheels with an incompatible mix.
REPO_DIR = Path('/kaggle/working/RNA-Seq-FS-Survey')
if REPO_DIR.exists():
    shutil.rmtree(REPO_DIR)
subprocess.run(['git', 'clone', '--branch', REPO_BRANCH, '--single-branch', REPO_URL, str(REPO_DIR)], check=True)
RUNTIME_IMPORTS = ('numpy', 'pandas', 'scipy', 'sklearn', 'tables', 'joblib', 'threadpoolctl')
try:
    for package in RUNTIME_IMPORTS:
        print(package, __import__(package).__version__)
except Exception as exc:
    raise RuntimeError('Kaggle numerical runtime is not coherent. Restart the Kaggle session, then rerun from Cell 1; do not run pip install -r requirements.txt.') from exc
WORK_ROOT.mkdir(parents=True, exist_ok=True)
for folder in ('experiment-data-perm', 'experiment-data-geo-target', 'split_manifests'):
    source = INPUT_V3_DATASET / folder
    target = WORK_ROOT / folder
    assert source.is_dir(), f'Missing Kaggle input: {source}'
    if not target.exists():
        target.symlink_to(source, target_is_directory=True)
os.environ['KLTN_DATA_ROOT'] = str(WORK_ROOT)
subprocess.run(['git', 'status', '--short'], cwd=REPO_DIR, check=True)
COMMIT = subprocess.check_output(['git', 'rev-parse', 'HEAD'], cwd=REPO_DIR, text=True).strip()
print('commit:', COMMIT)


In [ ]:
# Phase B with per-panel checkpoints. A stopped Kaggle session can resume safely.
os.chdir(REPO_DIR)
sys.path.insert(0, str(REPO_DIR))
import numpy as np
import pandas as pd
from threadpoolctl import threadpool_limits
from survey_v2 import config as C
from survey_v2.classify import evaluate_panel
from survey_v2.data import load_dataset, sha256
from survey_v2.e1_splits import validate_repeated_split_manifest
from scripts.run_e1_feature_selection import split_indices

def atomic_csv(frame, path):
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_suffix(path.suffix + '.tmp')
    frame.to_csv(temporary, index=False)
    temporary.replace(path)

def split_file(dataset):
    return dataset.data_dir / 'repeated_splits.csv'

def checkpoint_base(root, repeat, fold, method, k):
    return root / 'checkpoints' / f'r{repeat}_f{fold}_{method}_k{k}'

for COHORT in COHORTS:
    ds = load_dataset(WORK_ROOT, COHORT)
    manifest = pd.read_csv(split_file(ds))
    errors = validate_repeated_split_manifest(manifest, n_folds=C.N_FOLDS)
    assert not errors, errors
    panel_root = PANEL_INPUT_ROOT / CLASSIFICATION_GROUP / COHORT
    panels = pd.read_csv(panel_root / 'selected_genes.csv')
    selection = pd.read_csv(panel_root / 'selection_per_fold.csv')
    gene_index = {gene: i for i, gene in enumerate(ds.genes)}
    output_root = WORK_ROOT / 'results' / 'survey_v3' / 'E1_classification' / CLASSIFICATION_GROUP / COHORT

    for repeat in REPEATS:
        for fold in range(C.N_FOLDS):
            train, test = split_indices(ds, manifest, repeat, fold)
            seed = int(manifest[(manifest.repeat == repeat) & (manifest.fold == fold)].seed.iloc[0])
            for method in METHODS:
                for k in K_GRID:
                    selected = panels[(panels.repeat == repeat) & (panels.fold == fold) & (panels.method == method) & (panels.k == k)].sort_values('rank')
                    selection_row = selection[(selection.repeat == repeat) & (selection.fold == fold) & (selection.method == method) & (selection.k == k)]
                    # exact_k describes whether one sparse-path C yielded exactly k nonzero
                    # coefficients. A false value is valid: Phase A deterministically retains
                    # the k strongest coefficients from a solution with at least k nonzeros.
                    if len(selection_row) != 1 or len(selected) != k:
                        raise ValueError(f'Incomplete fixed panel: {COHORT}, r={repeat}, f={fold}, {method}, k={k}.')
                    genes = selected.gene.tolist()
                    if len(set(genes)) != k or any(g not in gene_index for g in genes):
                        raise ValueError(f'Panel has duplicate or unavailable genes: {COHORT}, r={repeat}, f={fold}, {method}, k={k}')
                    sparse_path_exact_k = str(selection_row.iloc[0].exact_k) == 'True'
                    sparse_path_n_filled = int(selection_row.iloc[0].n_filled)
                    base = checkpoint_base(output_root, repeat, fold, method, k)
                    metric_path, score_path = base.with_suffix('.metrics.csv'), base.with_suffix('.scores.csv')
                    if metric_path.is_file() and score_path.is_file():
                        continue
                    columns = [gene_index[g] for g in genes]
                    with threadpool_limits(1):
                        result = evaluate_panel(ds.X[train][:, columns], ds.y[train], ds.X[test][:, columns], ds.y[test], classifier='lr_l2', seed=C.seed_for(C.dataset_seed_id(COHORT), seed, fold, k), n_jobs=1)
                    atomic_csv(pd.DataFrame([{'cohort': COHORT, 'family': CLASSIFICATION_GROUP, 'repeat': repeat, 'fold': fold, 'seed': seed, 'method': method, 'k': k, 'classifier': result.classifier, 'auroc': result.auroc, 'bacc': result.bacc, 'mcc': result.mcc, 'threshold': result.threshold, 'hyperparam': result.hyperparam, 'n_selected': k, 'sparse_path_exact_k': sparse_path_exact_k, 'sparse_path_n_filled': sparse_path_n_filled}]), metric_path)
                    atomic_csv(pd.DataFrame({'cohort': COHORT, 'family': CLASSIFICATION_GROUP, 'repeat': repeat, 'fold': fold, 'method': method, 'k': k, 'classifier': result.classifier, 'sample_id': ds.sample_ids[test], 'y': ds.y[test], 'score': result.scores_test}), score_path)
                    print(f'{COHORT} {CLASSIFICATION_GROUP} r{repeat} f{fold} {method} k{k}: AUROC={result.auroc:.3f}', flush=True)


In [ ]:
# Consolidate every cohort only after all its requested checkpoints exist.
results_root = WORK_ROOT / 'results' / 'survey_v3' / 'E1_classification' / CLASSIFICATION_GROUP
expected = len(REPEATS) * C.N_FOLDS * len(METHODS) * len(K_GRID)
for COHORT in COHORTS:
    ds = load_dataset(WORK_ROOT, COHORT)
    panel_root = PANEL_INPUT_ROOT / CLASSIFICATION_GROUP / COHORT
    output_root = results_root / COHORT
    metric_files = sorted((output_root / 'checkpoints').glob('*.metrics.csv'))
    score_files = sorted((output_root / 'checkpoints').glob('*.scores.csv'))
    assert len(metric_files) == expected and len(score_files) == expected, (COHORT, len(metric_files), len(score_files), expected)
    per_fold = pd.concat([pd.read_csv(path) for path in metric_files], ignore_index=True)
    scores = pd.concat([pd.read_csv(path) for path in score_files], ignore_index=True)
    atomic_csv(per_fold, output_root / 'e1_per_fold.csv')
    atomic_csv(scores, output_root / 'e1_scores.csv')
    config = {'experiment': 'E1_phase_B_classification_v3', 'cohort': COHORT, 'family': CLASSIFICATION_GROUP, 'cohorts_requested': COHORTS, 'methods': METHODS, 'k_grid': K_GRID, 'repeats': REPEATS, 'classifier_main': 'lr_l2', 'selection_root': str(panel_root), 'selection_sha256': sha256(panel_root / 'selected_genes.csv'), 'split_sha256': sha256(split_file(ds)), 'commit_hash': COMMIT, 'traceable': True, 'wrapper_protocol_override': CLASSIFICATION_GROUP == 'wrappers'}
    (output_root / 'config.json').write_text(json.dumps(config, indent=2), encoding='utf-8')
archive = Path('/kaggle/working') / f'e1_classification_{CLASSIFICATION_GROUP}_all_cohorts.zip'
shutil.make_archive(str(archive.with_suffix('')), 'zip', root_dir=WORK_ROOT, base_dir='results/survey_v3/E1_classification')
print('DONE:', archive)
